<div text-align="center">
  <img src="https://raw.githubusercontent.com/FarnoushRJ/MambaLRP/main/assets/MambaLRP_logo.jpeg" width="1000"/>
</div>


<div text-align="center"><h1>🐍 MambaLRP is here! 🎉</h1>

Clone the repository and install MambaLRP.

In [1]:
# Cell 1: Clone
!git clone https://github.com/AdamBosch/MambaLRP.git || echo "Already cloned"

fatal: destination path 'MambaLRP' already exists and is not an empty directory.
Already cloned


In [2]:
# Cell 2: Pin torch first
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import subprocess
result = subprocess.run(['find', '/home/jovyan', '-name', 'torch', '-type', 'd'], 
                       capture_output=True, text=True)
print(result.stdout)

/home/jovyan/.local/lib/python3.10/site-packages/torch
/home/jovyan/.local/lib/python3.10/site-packages/torch/include/torch
/home/jovyan/.local/lib/python3.10/site-packages/torch/include/torch/csrc/api/include/torch



In [4]:
# Alternative Cell 3 - use pip's --find-links and pre-built wheels
!PYTHONPATH=/home/jovyan/.local/lib/python3.10/site-packages \
    pip install causal-conv1d==1.2.0.post2 --no-build-isolation --no-cache-dir -q
!PYTHONPATH=/home/jovyan/.local/lib/python3.10/site-packages \
    pip install mamba-ssm==1.2.0.post1 --no-build-isolation --no-cache-dir -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
# Cell 4: Write constraints file and install the rest
!echo "torch==2.1.0+cu118" > /tmp/constraints.txt
!pip install "numpy<2.0.0" "pyarrow<15.0.0" datasets==2.14.5 \
    transformers==4.40.1 captum==0.7.0 einops \
    -c /tmp/constraints.txt -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
# Cell 5: Install MambaLRP
!pip install ./MambaLRP --no-deps -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
# Cell 6: Restart kernel then verify
import torch
print(torch.__version__)       # should be 2.1.0+cu118
import causal_conv1d
import mamba_ssm
print("All imports OK")

2.1.0+cu118
All imports OK


Import necessary packages.

In [8]:
from transformers import MambaConfig, MambaForCausalLM, AutoTokenizer
import sys

from mamba_lrp.model.mamba_huggingface import ModifiedMambaForCausalLM
from mamba_lrp.model.utils import *
from mamba_lrp.lrp.utils import relevance_propagation
from mamba_lrp.dataset.general_dataset import get_sst_dataset, get_medbios_dataset
import torch
import numpy as np

## Load model

Load model and tokenizer.

In [9]:
!pip install gdown

# Import gdown
import gdown

# Define the file ID and the destination file name
file_id = '1RnIygUDodGeKPqbcEQTOYR5dztpF6X1b'  #SST-2
destination = 'mamba_sst2_weights.pt'

# Construct the URL
url = f'https://drive.google.com/uc?id={file_id}'

# Download the file
gdown.download(url, destination, quiet=False)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


Downloading...
From (original): https://drive.google.com/uc?id=1RnIygUDodGeKPqbcEQTOYR5dztpF6X1b
From (redirected): https://drive.google.com/uc?id=1RnIygUDodGeKPqbcEQTOYR5dztpF6X1b&confirm=t&uuid=ba853799-ca83-4539-ac50-e285c5db2d6f
To: /home/jovyan/mamba_sst2_weights.pt
100%|██████████| 517M/517M [00:06<00:00, 74.6MB/s] 


'mamba_sst2_weights.pt'

In [10]:
# Load tokenizer.
tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
tokenizer.eos_token = "<|endoftext|>"
tokenizer.bos_token = "<|startoftext|>"
tokenizer.pad_token = "<|pad|>"
tokenizer.unk_token = "<|unkown|>"
tokenizer.add_tokens(['<|unkown|>', '<|pad|>', "<|startoftext|>"], special_tokens=True)

# Load model.
model = MambaForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", use_cache=True)
resize_token_embeddings(model, len(tokenizer))
model.lm_head = torch.nn.Linear(768, 2, bias=True)

# Load the model's weights
model.load_state_dict(
    torch.load('mamba_sst2_weights.pt', map_location=torch.device('cpu')),
    strict=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Make model explainable.
modified_model = ModifiedMambaForCausalLM(model, is_fast_forward_available=False)
modified_model.eval()
model.backbone.embeddings.requires_grad = False
pretrained_embeddings = model.backbone.embeddings

/home/jovyan/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## Load dataset

Load SST-2 dataset.

In [11]:
validation_dataset = get_sst_dataset(
    tokenizer=tokenizer,
    truncation=False,
    max_length=None
    )

## Generate explanation

Generate explanation for one sample.

In [12]:
i = 413
input_ids = validation_dataset.__getitem__(i)['input_ids'].unsqueeze(0).to(device)
label = torch.tensor(validation_dataset.__getitem__(i)['label']).long().to(device)
idx = torch.where(input_ids == 0)[1] + 1
input_ids = input_ids[:, :idx]
embeddings = pretrained_embeddings(input_ids)

R, prediction = relevance_propagation(
    model=modified_model,
    embeddings=embeddings,
    targets=label,
    n_classes=2
    )

## Visualization

For simplicity, we use the visualization utilities in Captum to display the results.

In [13]:
from captum.attr import visualization as viz

In [14]:
tokens = []
for id in input_ids[0][1: -2]:
    tokens.append(tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens([id.item()])))
attributions = R[0][1: -2]
attributions = attributions / attributions.max()

In [15]:
# Visualize the attributions
viz.visualize_text([viz.VisualizationDataRecord(
    attributions,
    torch.max(model(input_ids).logits[:, -1, :], dim=1).values.item(),
    torch.argmax(model(input_ids).logits[:, -1, :], dim=1).item(),
    true_class=label.item(),
    attr_class=label.item(),
    attr_score=attributions.sum(),
    raw_input_ids=tokens,
    convergence_score=None
)])

True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,0 (2.86),0,1.67,at least one scene is so disgusting that viewers may be hard pressed to retain their lunch .


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,0 (2.86),0,1.67,at least one scene is so disgusting that viewers may be hard pressed to retain their lunch .


In [16]:
if isinstance(attributions, np.ndarray):
    attributions_tensor = torch.from_numpy(attributions)
    print("here")
else:
    attributions_tensor = attributions
    

sorted_indices = torch.argsort(attributions_tensor, descending=True)

here


In [17]:
def get_morf_curve_logits(model, input_ids, sorted_indices, target_label):
    scores = []
    temp_input_ids = input_ids.clone()
    
    # 0. Initial Logit
    with torch.no_grad():
        logits = model(temp_input_ids) 
        # Select the target class logit for the last token
        initial_logit = logits[0, -1, target_label].item()
        scores.append(initial_logit)

    # 1. Iteratively mask tokens and record the logit
    for idx in sorted_indices:
        # Map back to the correct position in input_ids
        temp_input_ids[0, idx + 1] = tokenizer.pad_token_id 
        
        with torch.no_grad():
            logits = model(temp_input_ids)
            logit = logits[0, -1, target_label].item()
            scores.append(logit)
            
    return scores
morf_values = get_morf_curve_logits(modified_model, input_ids, sorted_indices, label.item())

In [18]:
def get_morf_curve_batched(model, input_ids, embeddings, sorted_indices, target_label, chunk_size=8):
    seq_len = len(sorted_indices)
    
    # Work in embedding space, not token ID space
    base_embeddings = embeddings.clone().repeat(seq_len + 1, 1, 1)
    
    for i, idx in enumerate(sorted_indices):
        # Zero out the embedding at position idx+1 for all subsequent steps
        base_embeddings[i+1:, idx + 1, :] = 0.0
    
    all_logits = []
    with torch.no_grad():
        for i in range(0, base_embeddings.size(0), chunk_size):
            chunk = base_embeddings[i:i+chunk_size].to(device)
            logits = model(inputs_embeds=chunk)
            all_logits.extend(logits[:, -1, target_label].cpu().tolist())
            del chunk, logits
    return all_logits

In [19]:
import numpy as np
from tqdm import tqdm

num_samples = 100
all_delta_afs = []

print(f"Evaluating {num_samples} samples from SST-2...")

for i in tqdm(range(num_samples)):
    # 1. Get sample
    sample = validation_dataset.__getitem__(i)
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    label = torch.tensor(sample['label']).long().to(device)
    
    # 2. Trim padding for LRP
    idx = torch.where(input_ids == 0)[1] + 1
    input_ids = input_ids[:, :idx]
    
    with torch.no_grad():
        logits = modified_model(inputs_embeds=embeddings)
        pred_label = logits[:, -1].argmax(dim=-1)
    
    embeddings = pretrained_embeddings(input_ids)

    # 3. Generate LRP Attribution
    R, _ = relevance_propagation(
        model=modified_model,
        embeddings=embeddings,
        targets=pred_label,
        n_classes=2
    )
    
    # Extract relevance for tokens (excluding BOS/EOS as in Cell 14)
    # Note: R shape is [batch, seq_len], we take [0, 1:-2]
    attr = R[0][1:-2]
    if isinstance(attr, np.ndarray):
        attr = torch.from_numpy(attr)
    
    # 4. Calculate MoRF and LeRF
    # MoRF (Most Relevant First)
    idx_morf = torch.argsort(attr, descending=True)
    morf_probs = get_morf_curve_batched(modified_model, input_ids, embeddings, idx_morf, pred_label.item())
    af_morf = np.trapz(morf_probs) / len(morf_probs)
    
    # LeRF (Least Relevant First)
    idx_lerf = torch.argsort(attr, descending=False)
    lerf_probs = get_morf_curve_batched(modified_model, input_ids, embeddings, idx_lerf, pred_label.item())
    af_lerf = np.trapz(lerf_probs) / len(lerf_probs)
    
    all_delta_afs.append(af_lerf - af_morf)

# Final Result
mean_delta_af = np.mean(all_delta_afs)
print(f"\nMean Delta AF over {num_samples} samples: {mean_delta_af:.4f}")
print(f"Paper Reference (Mamba-130M SST-2): 1.978")

Evaluating 100 samples from SST-2...


100%|██████████| 100/100 [01:19<00:00,  1.26it/s]


Mean Delta AF over 100 samples: 1.3267
Paper Reference (Mamba-130M SST-2): 1.978
